In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
import shutil, os
if not os.path.exists('/content/tray_food'):
    shutil.copytree('/content/drive/MyDrive/tray_food', '/content/tray_food')

In [8]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, GlobalAveragePooling2D, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2

In [9]:
train_dir = "/content/tray_food/"
img_size = 128
batch_size = 16

In [10]:
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.15,
    horizontal_flip=True,
    brightness_range=[0.85, 1.15],
    fill_mode="nearest",
    validation_split=0.2)

validation_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    validation_split=0.2
)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode="categorical",
    shuffle=True,
    seed=42,
    subset="training")

validation_generator = validation_datagen.flow_from_directory(
    train_dir,
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode="categorical",
    shuffle=False,
    seed=42,
    subset="validation")

Found 2412 images belonging to 16 classes.
Found 595 images belonging to 16 classes.


/usr/local/lib/python3.12/dist-packages/keras/src/legacy/preprocessing/image.py:149: UserWarning: Using ".tiff" files with multiple bands will cause distortion. Please verify your output.
  warnings.warn(


In [11]:
model = Sequential([
    Conv2D(32, (3,3), padding='same', activation='relu', input_shape=(128,128,3)),
    BatchNormalization(),
    Conv2D(32, (3,3), padding='same', activation='relu'),
    BatchNormalization(),
    MaxPooling2D(),
    Dropout(0.25),

    Conv2D(64, (3,3), padding='same', activation='relu'),
    BatchNormalization(),
    Conv2D(64, (3,3), padding='same', activation='relu'),
    BatchNormalization(),
    MaxPooling2D(),
    Dropout(0.25),

    Conv2D(128, (3,3), padding='same', activation='relu', kernel_regularizer=l2(1e-4)),
    BatchNormalization(),
    Conv2D(128, (3,3), padding='same', activation='relu', kernel_regularizer=l2(1e-4)),
    BatchNormalization(),
    MaxPooling2D(),
    Dropout(0.3),

    GlobalAveragePooling2D(),

    Dense(256, activation='relu', kernel_regularizer=l2(1e-4)),
    Dropout(0.4),

    Dense(16, activation='softmax')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [12]:
epochs = 80

model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy'])

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 128, 128, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128, 128, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 128, 128, 32)   │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 128, 128, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 64, 64, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 64, 64, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 64, 64, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 64, 64, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 32, 32, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 32, 32, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 32, 32, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 32, 32, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼─────────────

 Total params: 325,936 (1.24 MB)

 Trainable params: 325,040 (1.24 MB)

 Non-trainable params: 896 (3.50 KB)

In [13]:
callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=15, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1),
    ModelCheckpoint('best_model.h5', monitor='val_accuracy', save_best_only=True)]

history = model.fit(train_generator, validation_data=validation_generator, epochs=epochs, callbacks=callbacks)

Epoch 1/80
151/151 ━━━━━━━━━━━━━━━━━━━━ 0s 339ms/step - accuracy: 0.2005 - loss: 2.4835

/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


151/151 ━━━━━━━━━━━━━━━━━━━━ 77s 402ms/step - accuracy: 0.2297 - loss: 2.2871 - val_accuracy: 0.0605 - val_loss: 6.5671 - learning_rate: 0.0010
Epoch 2/80
151/151 ━━━━━━━━━━━━━━━━━━━━ 0s 263ms/step - accuracy: 0.3031 - loss: 2.0300

151/151 ━━━━━━━━━━━━━━━━━━━━ 47s 308ms/step - accuracy: 0.3209 - loss: 1.9944 - val_accuracy: 0.0975 - val_loss: 5.7519 - learning_rate: 0.0010
Epoch 3/80
151/151 ━━━━━━━━━━━━━━━━━━━━ 0s 265ms/step - accuracy: 0.3568 - loss: 1.8959

151/151 ━━━━━━━━━━━━━━━━━━━━ 46s 303ms/step - accuracy: 0.3619 - loss: 1.8657 - val_accuracy: 0.1513 - val_loss: 4.1530 - learning_rate: 0.0010
Epoch 4/80
151/151 ━━━━━━━━━━━━━━━━━━━━ 0s 267ms/step - accuracy: 0.4081 - loss: 1.8043

...

151/151 ━━━━━━━━━━━━━━━━━━━━ 46s 302ms/step - accuracy: 0.8279 - loss: 0.5598 - val_accuracy: 0.7328 - val_loss: 0.8663 - learning_rate: 1.2500e-04
Epoch 52/80
151/151 ━━━━━━━━━━━━━━━━━━━━ 0s 264ms/step - accuracy: 0.8185 - loss: 0.5476

151/151 ━━━━━━━━━━━━━━━━━━━━ 46s 308ms/step - accuracy: 0.8172 - loss: 0.5559 - val_accuracy: 0.7429 - val_loss: 0.8536 - learning_rate: 1.2500e-04
Epoch 53/80
151/151 ━━━━━━━━━━━━━━━━━━━━ 45s 300ms/step - accuracy: 0.8201 - loss: 0.5600 - val_accuracy: 0.7412 - val_loss: 0.8686 - learning_rate: 1.2500e-04
Epoch 54/80
151/151 ━━━━━━━━━━━━━━━━━━━━ 45s 300ms/step - accuracy: 0.8292 - loss: 0.5334 - val_accuracy: 0.7277 - val_loss: 0.9200 - learning_rate: 1.2500e-04
Epoch 55/80
151/151 ━━━━━━━━━━━━━━━━━━━━ 45s 299ms/step - accuracy: 0.8362 - loss: 0.5333 - val_accuracy: 0.6723 - val_loss: 1.1431 - learning_rate: 1.2500e-04
Epoch 56/80
151/151 ━━━━━━━━━━━━━━━━━━━━ 0s 264ms/step - accuracy: 0.8181 - loss: 0.5334

151/151 ━━━━━━━━━━━━━━━━━━━━ 46s 308ms/step - accuracy: 0.8064 - loss: 0.5665 - val_accuracy: 0.7496 - val_loss: 0.8515 - learning_rate: 1.2500e-04
Epoch 57/80
151/151 ━━━━━━━━━━━━━━━━━━━━ 45s 298ms/step - accuracy: 0.8333 - loss: 0.5331 - val_accuracy: 0.7143 - val_loss: 0.9557 - learning_rate: 1.2500e-04
Epoch 58/80
151/151 ━━━━━━━━━━━━━━━━━━━━ 45s 297ms/step - accuracy: 0.8292 - loss: 0.5142 - val_accuracy: 0.7227 - val_loss: 0.9991 - learning_rate: 1.2500e-04
Epoch 59/80
151/151 ━━━━━━━━━━━━━━━━━━━━ 45s 299ms/step - accuracy: 0.8304 - loss: 0.5291 - val_accuracy: 0.7042 - val_loss: 1.0127 - learning_rate: 1.2500e-04
Epoch 60/80
151/151 ━━━━━━━━━━━━━━━━━━━━ 50s 327ms/step - accuracy: 0.8197 - loss: 0.5586 - val_accuracy: 0.7227 - val_loss: 0.9438 - learning_rate: 1.2500e-04
Epoch 61/80
151/151 ━━━━━━━━━━━━━━━━━━━━ 0s 261ms/step - accuracy: 0.8478 - loss: 0.5033
Epoch 61: ReduceLROnPlateau reducing learning rate to 6.25000029685907e-05.
151/151 ━━━━━━━━━━━━━━━━━━━━ 45s 298ms/step - a

151/151 ━━━━━━━━━━━━━━━━━━━━ 45s 298ms/step - accuracy: 0.8333 - loss: 0.5177 - val_accuracy: 0.7513 - val_loss: 0.8461 - learning_rate: 6.2500e-05
Epoch 63/80
151/151 ━━━━━━━━━━━━━━━━━━━━ 46s 303ms/step - accuracy: 0.8570 - loss: 0.4799 - val_accuracy: 0.7462 - val_loss: 0.8817 - learning_rate: 6.2500e-05
Epoch 64/80
151/151 ━━━━━━━━━━━━━━━━━━━━ 0s 255ms/step - accuracy: 0.8377 - loss: 0.5061

151/151 ━━━━━━━━━━━━━━━━━━━━ 45s 298ms/step - accuracy: 0.8499 - loss: 0.4783 - val_accuracy: 0.7563 - val_loss: 0.8279 - learning_rate: 6.2500e-05
Epoch 65/80
151/151 ━━━━━━━━━━━━━━━━━━━━ 46s 303ms/step - accuracy: 0.8362 - loss: 0.4933 - val_accuracy: 0.7143 - val_loss: 0.9600 - learning_rate: 6.2500e-05
Epoch 66/80
151/151 ━━━━━━━━━━━━━━━━━━━━ 46s 307ms/step - accuracy: 0.8387 - loss: 0.5002 - val_accuracy: 0.7395 - val_loss: 0.8850 - learning_rate: 6.2500e-05
Epoch 67/80
151/151 ━━━━━━━━━━━━━━━━━━━━ 46s 304ms/step - accuracy: 0.8624 - loss: 0.4610 - val_accuracy: 0.7076 - val_loss: 0.9864 - learning_rate: 6.2500e-05
Epoch 68/80
151/151 ━━━━━━━━━━━━━━━━━━━━ 46s 306ms/step - accuracy: 0.8507 - loss: 0.4836 - val_accuracy: 0.7126 - val_loss: 1.0356 - learning_rate: 6.2500e-05
Epoch 69/80
151/151 ━━━━━━━━━━━━━━━━━━━━ 0s 265ms/step - accuracy: 0.8564 - loss: 0.4605
Epoch 69: ReduceLROnPlateau reducing learning rate to 3.125000148429535e-05.
151/151 ━━━━━━━━━━━━━━━━━━━━ 46s 303ms/step - 

In [14]:
model.save('food_cnn.h5')
print('Model saved: food_cnn.h5')

class_names = [k for k, v in sorted(train_generator.class_indices.items(), key=lambda x: x[1])]
with open('class_names.txt', 'w') as f:
    f.write('\n'.join(class_names))
print('Classes saved: class_names.txt')
print(class_names)

Model saved: food_cnn.h5
Classes saved: class_names.txt
['canh chua có cá', 'canh chua không cá', 'canh rau cải thảo', 'canh rau muống', 'cá hú kho', 'cơm', 'rau xào củ sắn', 'rau xào lagim', 'rau xào đậu que', 'rau xào đậu đũa', 'sườn nướng', 'thịt kho', 'thịt kho trứng', 'trứng chiên', 'trứng chiên thịt', 'đậu hũ sốt cà']
